In [15]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler
from sklearn.model_selection import train_test_split
import pickle
from sklearn.decomposition import PCA

In [16]:
pd.set_option('display.max_columns', None)  # or 1000
# pd.set_option('display.max_rows', None)  # or 1000

In [14]:
nbin = 5

# Iris

In [4]:
column_names = ['sepal.length', 'sepal.width', 'petal.length', 'petal.width', 'class']

iris = pd.read_csv('dataset/iris.data', sep=",", header=None, names=column_names, engine='python')

In [5]:
iris.head()

,sepal.length,sepal.width,petal.length,petal.width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [591]:
# iris = iris[iris['class'] != 'Iris-virginica']

In [6]:
iris['class'] = iris['class'].astype('category')

In [7]:
iris.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
sepal.length    150 non-null float64
sepal.width     150 non-null float64
petal.length    150 non-null float64
petal.width     150 non-null float64
class           150 non-null category
dtypes: category(1), float64(4)
memory usage: 5.1 KB


In [8]:
iris.describe()

,sepal.length,sepal.width,petal.length,petal.width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.054000,3.758667,1.198667
std,0.828066,0.433594,1.764420,0.763161
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


In [9]:
iris.select_dtypes('category').describe()

,class
count,150
unique,3
top,Iris-virginica
freq,50


In [10]:
display(iris.corr())

,sepal.length,sepal.width,petal.length,petal.width
sepal.length,1.000000,-0.109369,0.871754,0.817954
sepal.width,-0.109369,1.000000,-0.420516,-0.356544
petal.length,0.871754,-0.420516,1.000000,0.962757
petal.width,0.817954,-0.356544,0.962757,1.000000


In [11]:
# Features
iris_data = iris.drop(columns = ['class'])
iris_label = iris['class']

iris_non_cat_col = iris_data.select_dtypes(exclude = 'category').columns
for c in iris_non_cat_col:
    iris_data[c] = pd.cut(iris_data[c], bins=nbin, labels=range(nbin))
    
iris_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
sepal.length    150 non-null category
sepal.width     150 non-null category
petal.length    150 non-null category
petal.width     150 non-null category
dtypes: category(4)
memory usage: 1.2 KB


In [12]:
iris_data[iris_data.isnull().any(axis=1)]

,sepal.length,sepal.width,petal.length,petal.width


In [13]:
iris_data.head()

,sepal.length,sepal.width,petal.length,petal.width
0,1,3,0,0
1,0,2,0,0
2,0,2,0,0
3,0,2,0,0
4,0,3,0,0


In [14]:
iris_label.head()

0    Iris-setosa
1    Iris-setosa
2    Iris-setosa
3    Iris-setosa
4    Iris-setosa
Name: class, dtype: category
Categories (3, object): [Iris-setosa, Iris-versicolor, Iris-virginica]

In [15]:
iris_label.cat.codes

0      0
1      0
2      0
3      0
4      0
      ..
145    2
146    2
147    2
148    2
149    2
Length: 150, dtype: int8

In [16]:
iris_data.shape

(150, 4)

In [17]:
with open('dataset/iris-bin{}.pkl'.format(nbin), 'wb') as f:
    pickle.dump((iris_data, pd.Series(iris_label.cat.codes, name='class')), f)

# Diabetic

In [72]:
# https://www.hindawi.com/journals/bmri/2014/781670/tab1/
# http://downloads.hindawi.com/journals/bmri/2014/781670.pdf
diab = pd.read_csv('dataset/diabetic.csv', sep=",", header=0, engine='python', na_values=['?'])

In [73]:
diab.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [74]:
diab.columns = diab.columns[:-1].tolist() + ['class']

In [75]:
diab['class']

0          NO
1         >30
2          NO
3          NO
4          NO
         ... 
101761    >30
101762     NO
101763     NO
101764     NO
101765     NO
Name: class, Length: 101766, dtype: object

In [76]:
diab['class'] = diab['class'].astype('category')

In [77]:
diab.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
encounter_id                101766 non-null int64
patient_nbr                 101766 non-null int64
race                        99493 non-null object
gender                      101766 non-null object
age                         101766 non-null object
weight                      3197 non-null object
admission_type_id           101766 non-null int64
discharge_disposition_id    101766 non-null int64
admission_source_id         101766 non-null int64
time_in_hospital            101766 non-null int64
payer_code                  61510 non-null object
medical_specialty           51817 non-null object
num_lab_procedures          101766 non-null int64
num_procedures              101766 non-null int64
num_medications             101766 non-null int64
number_outpatient           101766 non-null int64
number_emergency            101766 non-null int64
number_inpatient            101766 non

In [78]:
diab_obj_col = diab.select_dtypes('object').columns
for c in diab_obj_col:
    diab[c] = diab[c].astype('category')

In [83]:
diab_obj_col = diab.select_dtypes('category').columns
print([len(diab[c].cat.categories) for c in diab_obj_col])

[5, 3, 10, 9, 17, 72, 716, 748, 789, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 2, 4, 4, 4, 4, 2, 3, 1, 1, 4, 4, 2, 2, 2, 2, 2, 2, 3]


In [87]:
diab = diab.drop(columns = ['diag_1','diag_2','diag_3','medical_specialty'])

In [89]:
diab.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 46 columns):
encounter_id                101766 non-null int64
patient_nbr                 101766 non-null int64
race                        99493 non-null category
gender                      101766 non-null category
age                         101766 non-null category
weight                      3197 non-null category
admission_type_id           101766 non-null int64
discharge_disposition_id    101766 non-null int64
admission_source_id         101766 non-null int64
time_in_hospital            101766 non-null int64
payer_code                  61510 non-null category
num_lab_procedures          101766 non-null int64
num_procedures              101766 non-null int64
num_medications             101766 non-null int64
number_outpatient           101766 non-null int64
number_emergency            101766 non-null int64
number_inpatient            101766 non-null int64
number_diagnoses            

In [90]:
diab.describe()

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,1.017660e+05,1.017660e+05,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,5.433040e+07,2.024006,3.715642,5.754437,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,1.026403e+08,3.869636e+07,1.445403,5.280166,4.064081,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.252200e+04,1.350000e+02,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,8.496119e+07,2.341322e+07,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.523890e+08,4.550514e+07,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,2.302709e+08,8.754595e+07,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,4.438672e+08,1.895026e+08,8.000000,28.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


In [91]:
diab.select_dtypes('category').describe()

,race,gender,age,weight,payer_code,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,class
count,99493,101766,101766,3197,61510,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766
unique,5,3,10,9,17,4,4,4,4,4,4,4,2,4,4,2,4,4,4,4,2,3,1,1,4,4,2,2,2,2,2,2,3
top,Caucasian,Female,[70-80),[75-100),MC,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
freq,76099,54708,26068,1336,32439,96420,84748,81778,100227,101063,101680,96575,101765,89080,91116,101743,94438,95401,101458,101728,101763,101727,101766,101766,47383,101060,101753,101765,101764,101765,54755,78363,54864


In [92]:
display(diab.corr())

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
encounter_id,1.000000,0.512028,-0.158961,-0.132876,-0.112402,-0.062221,-0.026062,-0.014225,0.076113,0.103756,0.082803,0.030962,0.265149
patient_nbr,0.512028,1.000000,-0.011128,-0.136814,-0.032568,-0.024092,0.015946,-0.015570,0.020665,0.103379,0.062352,0.012480,0.226847
admission_type_id,-0.158961,-0.011128,1.000000,0.083483,0.106654,-0.012500,-0.143713,0.129888,0.079535,0.026511,-0.019116,-0.038161,-0.117126
discharge_disposition_id,-0.132876,-0.136814,0.083483,1.000000,0.018193,0.162748,0.023415,0.015921,0.108753,-0.008715,-0.024471,0.020787,0.046891
admission_source_id,-0.112402,-0.032568,0.106654,0.018193,1.000000,-0.006965,0.048885,-0.135400,-0.054533,0.027244,0.059892,0.036314,0.072114
time_in_hospital,-0.062221,-0.024092,-0.012500,0.162748,-0.006965,1.000000,0.318450,0.191472,0.466135,-0.008916,-0.009681,0.073623,0.220186
num_lab_procedures,-0.026062,0.015946,-0.143713,0.023415,0.048885,0.318450,1.000000,0.058066,0.268161,-0.007602,-0.002279,0.039231,0.152773
num_procedures,-0.014225,-0.015570,0.129888,0.015921,-0.135400,0.191472,0.058066,1.000000,0.385767,-0.024819,-0.038179,-0.066236,0.073734
num_medications,0.076113,0.020665,0.079535,0.108753,-0.054533,0.466135,0.268161,0.385767,1.000000,0.045197,0.013180,0.064194,0.261526
number_outpatient,0.103756,0.103379,0.026511,-0.008715,0.027244,-0.008916,-0.007602,-0.024819,0.045197,1.000000,0.091459,0.107338,0.094152


In [107]:
# Features
diab_data = diab.drop(columns = ['class'])
diab_label = diab['class']

diab_non_cat_col = diab_data.select_dtypes(exclude = 'category').columns
for c in diab_non_cat_col:
    diab_data[c] = pd.cut(diab_data[c], bins=nbin, labels=range(nbin))
    
diab_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 45 columns):
encounter_id                101766 non-null category
patient_nbr                 101766 non-null category
race                        99493 non-null category
gender                      101766 non-null category
age                         101766 non-null category
weight                      3197 non-null category
admission_type_id           101766 non-null category
discharge_disposition_id    101766 non-null category
admission_source_id         101766 non-null category
time_in_hospital            101766 non-null category
payer_code                  61510 non-null category
num_lab_procedures          101766 non-null category
num_procedures              101766 non-null category
num_medications             101766 non-null category
number_outpatient           101766 non-null category
number_emergency            101766 non-null category
number_inpatient            101766 non-null c

In [108]:
diab_data[diab_data.isnull().any(axis=1)]

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
0,0,0,Caucasian,Female,[0-10),NaN,3,4,0,0,NaN,1,0,0,0,0,0,0,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No
1,0,1,Caucasian,Female,[10-20),NaN,0,0,1,0,NaN,2,0,1,0,0,0,2,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes
2,0,2,AfricanAmerican,Female,[20-30),NaN,0,0,1,0,NaN,0,4,0,0,0,0,1,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes
3,0,2,Caucasian,Male,[30-40),NaN,0,0,1,0,NaN,1,0,0,0,0,0,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes
4,0,1,Caucasian,Male,[40-50),NaN,0,0,1,0,NaN,1,0,0,0,0,0,1,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,4,2,AfricanAmerican,Male,[70-80),NaN,0,0,1,0,MC,1,0,0,0,0,0,2,None,>8,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes
101762,4,1,AfricanAmerican,Female,[80-90),NaN,0,0,0,1,MC,1,2,1,0,0,0,2,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes
101763,4,1,Caucasian,Male,[70-80),NaN,0,0,1,0,MC,1,0,0,0,0,0,3,None,None,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes
101764,4,0,Caucasian,Female,[80-90),NaN,0,0,1,3,MC,1,1,1,0,0,0,2,None,None,No,No,No,No,No,No,Steady,No,No,Steady,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes


In [54]:
diab_data.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
0,0,0,Caucasian,Female,[0-10),?,3,4,0,0,?,Pediatrics-Endocrinology,1,0,0,0,0,0,250.83,?,?,0,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No
1,0,1,Caucasian,Female,[10-20),?,0,0,1,0,?,?,2,0,1,0,0,0,276,250.01,255,2,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes
2,0,2,AfricanAmerican,Female,[20-30),?,0,0,1,0,?,?,0,4,0,0,0,0,648,250,V27,1,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes
3,0,2,Caucasian,Male,[30-40),?,0,0,1,0,?,?,1,0,0,0,0,0,8,250.43,403,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes
4,0,1,Caucasian,Male,[40-50),?,0,0,1,0,?,?,1,0,0,0,0,0,197,157,250,1,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes


In [109]:
diab_label.head()

0     NO
1    >30
2     NO
3     NO
4     NO
Name: class, dtype: category
Categories (3, object): [<30, >30, NO]

In [110]:
diab_label.cat.codes

0         2
1         1
2         2
3         2
4         2
         ..
101761    1
101762    2
101763    2
101764    2
101765    2
Length: 101766, dtype: int8

In [111]:
diab_data.shape

(101766, 45)

In [112]:
with open('dataset/diab-bin{}.pkl'.format(nbin), 'wb') as f:
    pickle.dump((diab_data, pd.Series(diab_label.cat.codes, name='class')), f)

# Shoppers

In [113]:
shoppers = pd.read_csv('dataset/online_shoppers_intention.csv', sep=",", header=0, engine='python', na_values=['?'])

In [114]:
shoppers.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [115]:
shoppers.columns = shoppers.columns[:-1].tolist() + ['class']

In [116]:
shoppers['class']

0        False
1        False
2        False
3        False
4        False
         ...  
12325    False
12326    False
12327    False
12328    False
12329    False
Name: class, Length: 12330, dtype: bool

In [122]:
shoppers['class'] = shoppers['class'].astype('category')

In [117]:
shoppers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
Administrative             12330 non-null int64
Administrative_Duration    12330 non-null float64
Informational              12330 non-null int64
Informational_Duration     12330 non-null float64
ProductRelated             12330 non-null int64
ProductRelated_Duration    12330 non-null float64
BounceRates                12330 non-null float64
ExitRates                  12330 non-null float64
PageValues                 12330 non-null float64
SpecialDay                 12330 non-null float64
Month                      12330 non-null object
OperatingSystems           12330 non-null int64
Browser                    12330 non-null int64
Region                     12330 non-null int64
TrafficType                12330 non-null int64
VisitorType                12330 non-null object
Weekend                    12330 non-null bool
class                      12330 non-null bool
dtypes: bool(

In [119]:
shoppers_obj_col = ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']
for c in shoppers_obj_col:
    shoppers[c] = shoppers[c].astype('category')

In [120]:
shoppers_obj_col = shoppers.select_dtypes('category').columns
print([len(shoppers[c].cat.categories) for c in shoppers_obj_col])

[10, 8, 13, 9, 20, 3, 2]


In [87]:
# shoppers = shoppers.drop(columns = ['diag_1','diag_2','diag_3','medical_specialty'])

In [123]:
shoppers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
Administrative             12330 non-null int64
Administrative_Duration    12330 non-null float64
Informational              12330 non-null int64
Informational_Duration     12330 non-null float64
ProductRelated             12330 non-null int64
ProductRelated_Duration    12330 non-null float64
BounceRates                12330 non-null float64
ExitRates                  12330 non-null float64
PageValues                 12330 non-null float64
SpecialDay                 12330 non-null float64
Month                      12330 non-null category
OperatingSystems           12330 non-null category
Browser                    12330 non-null category
Region                     12330 non-null category
TrafficType                12330 non-null category
VisitorType                12330 non-null category
Weekend                    12330 non-null category
class                      12330 non-nul

In [124]:
shoppers.describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157213,0.016813,0.050000,0.000000,0.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000


In [125]:
shoppers.select_dtypes('category').describe()

,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,class
count,12330,12330,12330,12330,12330,12330,12330,12330
unique,10,8,13,9,20,3,2,2
top,May,2,2,1,2,Returning_Visitor,False,False
freq,3364,6601,7961,4780,3913,10551,9462,10422


In [126]:
display(shoppers.corr())

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay
Administrative,1.000000,0.601583,0.376850,0.255848,0.431119,0.373939,-0.223563,-0.316483,0.098990,-0.094778
Administrative_Duration,0.601583,1.000000,0.302710,0.238031,0.289087,0.355422,-0.144170,-0.205798,0.067608,-0.073304
Informational,0.376850,0.302710,1.000000,0.618955,0.374164,0.387505,-0.116114,-0.163666,0.048632,-0.048219
Informational_Duration,0.255848,0.238031,0.618955,1.000000,0.280046,0.347364,-0.074067,-0.105276,0.030861,-0.030577
ProductRelated,0.431119,0.289087,0.374164,0.280046,1.000000,0.860927,-0.204578,-0.292526,0.056282,-0.023958
ProductRelated_Duration,0.373939,0.355422,0.387505,0.347364,0.860927,1.000000,-0.184541,-0.251984,0.052823,-0.036380
BounceRates,-0.223563,-0.144170,-0.116114,-0.074067,-0.204578,-0.184541,1.000000,0.913004,-0.119386,0.072702
ExitRates,-0.316483,-0.205798,-0.163666,-0.105276,-0.292526,-0.251984,0.913004,1.000000,-0.174498,0.102242
PageValues,0.098990,0.067608,0.048632,0.030861,0.056282,0.052823,-0.119386,-0.174498,1.000000,-0.063541
SpecialDay,-0.094778,-0.073304,-0.048219,-0.030577,-0.023958,-0.036380,0.072702,0.102242,-0.063541,1.000000


In [127]:
# Features
shoppers_data = shoppers.drop(columns = ['class'])
shoppers_label = shoppers['class']

shoppers_non_cat_col = shoppers_data.select_dtypes(exclude = 'category').columns
for c in shoppers_non_cat_col:
    shoppers_data[c] = pd.cut(shoppers_data[c], bins=nbin, labels=range(nbin))
    
shoppers_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 17 columns):
Administrative             12330 non-null category
Administrative_Duration    12330 non-null category
Informational              12330 non-null category
Informational_Duration     12330 non-null category
ProductRelated             12330 non-null category
ProductRelated_Duration    12330 non-null category
BounceRates                12330 non-null category
ExitRates                  12330 non-null category
PageValues                 12330 non-null category
SpecialDay                 12330 non-null category
Month                      12330 non-null category
OperatingSystems           12330 non-null category
Browser                    12330 non-null category
Region                     12330 non-null category
TrafficType                12330 non-null category
VisitorType                12330 non-null category
Weekend                    12330 non-null category
dtypes: category(17)
mem

In [128]:
shoppers_data[shoppers_data.isnull().any(axis=1)]

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend


In [129]:
shoppers_data.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend
0,0,0,0,0,0,0,4,4,0,0,Feb,1,1,1,1,Returning_Visitor,False
1,0,0,0,0,0,0,0,2,0,0,Feb,2,2,1,2,Returning_Visitor,False
2,0,0,0,0,0,0,4,4,0,0,Feb,4,1,9,3,Returning_Visitor,False
3,0,0,0,0,0,0,1,3,0,0,Feb,3,2,2,4,Returning_Visitor,False
4,0,0,0,0,0,0,0,1,0,0,Feb,3,3,1,4,Returning_Visitor,True


In [130]:
shoppers_label.head()

0    False
1    False
2    False
3    False
4    False
Name: class, dtype: category
Categories (2, object): [False, True]

In [131]:
shoppers_label.cat.codes

0        0
1        0
2        0
3        0
4        0
        ..
12325    0
12326    0
12327    0
12328    0
12329    0
Length: 12330, dtype: int8

In [132]:
shoppers_data.shape

(12330, 17)

In [133]:
with open('dataset/shoppers-bin{}.pkl'.format(nbin), 'wb') as f:
    pickle.dump((shoppers_data, pd.Series(shoppers_label.cat.codes, name='class')), f)

# Default

In [138]:
# default = pd.read_csv('dataset/default of credit card clients.xls', sep=",", header=0, engine='python', na_values=['?'])
default = pd.read_excel('dataset/default of credit card clients.xls', skiprows=1)

In [139]:
default.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [140]:
default.columns = default.columns[:-1].tolist() + ['class']

In [141]:
default['class']

0        1
1        1
2        0
3        0
4        0
        ..
29995    0
29996    0
29997    1
29998    1
29999    1
Name: class, Length: 30000, dtype: int64

In [143]:
default['class'] = default['class'].astype('category')

In [144]:
default.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
ID           30000 non-null int64
LIMIT_BAL    30000 non-null int64
SEX          30000 non-null int64
EDUCATION    30000 non-null int64
MARRIAGE     30000 non-null int64
AGE          30000 non-null int64
PAY_0        30000 non-null int64
PAY_2        30000 non-null int64
PAY_3        30000 non-null int64
PAY_4        30000 non-null int64
PAY_5        30000 non-null int64
PAY_6        30000 non-null int64
BILL_AMT1    30000 non-null int64
BILL_AMT2    30000 non-null int64
BILL_AMT3    30000 non-null int64
BILL_AMT4    30000 non-null int64
BILL_AMT5    30000 non-null int64
BILL_AMT6    30000 non-null int64
PAY_AMT1     30000 non-null int64
PAY_AMT2     30000 non-null int64
PAY_AMT3     30000 non-null int64
PAY_AMT4     30000 non-null int64
PAY_AMT5     30000 non-null int64
PAY_AMT6     30000 non-null int64
class        30000 non-null category
dtypes: category(1), int64(24)
memory 

In [146]:
default_obj_col = ['SEX', 'EDUCATION', 'MARRIAGE']
for c in default_obj_col:
    default[c] = default[c].astype('category')

In [147]:
default_obj_col = default.select_dtypes('category').columns
print([len(default[c].cat.categories) for c in default_obj_col])

[2, 7, 4, 2]


In [87]:
# default = default.drop(columns = ['diag_1','diag_2','diag_3','medical_specialty'])

In [148]:
default.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
ID           30000 non-null int64
LIMIT_BAL    30000 non-null int64
SEX          30000 non-null category
EDUCATION    30000 non-null category
MARRIAGE     30000 non-null category
AGE          30000 non-null int64
PAY_0        30000 non-null int64
PAY_2        30000 non-null int64
PAY_3        30000 non-null int64
PAY_4        30000 non-null int64
PAY_5        30000 non-null int64
PAY_6        30000 non-null int64
BILL_AMT1    30000 non-null int64
BILL_AMT2    30000 non-null int64
BILL_AMT3    30000 non-null int64
BILL_AMT4    30000 non-null int64
BILL_AMT5    30000 non-null int64
BILL_AMT6    30000 non-null int64
PAY_AMT1     30000 non-null int64
PAY_AMT2     30000 non-null int64
PAY_AMT3     30000 non-null int64
PAY_AMT4     30000 non-null int64
PAY_AMT5     30000 non-null int64
PAY_AMT6     30000 non-null int64
class        30000 non-null category
dtypes: category(4), int64(21

In [149]:
default.describe()

,ID,LIMIT_BAL,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,-0.266200,-0.291100,51223.330900,49179.075167,4.701315e+04,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567
std,8660.398374,129747.661567,9.217904,1.123802,1.197186,1.196868,1.169139,1.133187,1.149988,73635.860576,71173.768783,6.934939e+04,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775
min,1.000000,10000.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-165580.000000,-69777.000000,-1.572640e+05,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,3558.750000,2984.750000,2.666250e+03,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000
50%,15000.500000,140000.000000,34.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,22381.500000,21200.000000,2.008850e+04,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000
75%,22500.250000,240000.000000,41.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,67091.000000,64006.250000,6.016475e+04,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000
max,30000.000000,1000000.000000,79.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,964511.000000,983931.000000,1.664089e+06,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000


In [150]:
default.select_dtypes('category').describe()

,SEX,EDUCATION,MARRIAGE,class
count,30000,30000,30000,30000
unique,2,7,4,2
top,2,2,2,0
freq,18112,14030,15964,23364


In [151]:
display(default.corr())

,ID,LIMIT_BAL,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
ID,1.000000,0.026179,0.018678,-0.030575,-0.011215,-0.018494,-0.002735,-0.022199,-0.020270,0.019389,0.017982,0.024354,0.040351,0.016705,0.016730,0.009742,0.008406,0.039151,0.007793,0.000652,0.003000
LIMIT_BAL,0.026179,1.000000,0.144713,-0.271214,-0.296382,-0.286123,-0.267460,-0.249411,-0.235195,0.285430,0.278314,0.283236,0.293988,0.295562,0.290389,0.195236,0.178408,0.210167,0.203242,0.217202,0.219595
AGE,0.018678,0.144713,1.000000,-0.039447,-0.050148,-0.053048,-0.049722,-0.053826,-0.048773,0.056239,0.054283,0.053710,0.051353,0.049345,0.047613,0.026147,0.021785,0.029247,0.021379,0.022850,0.019478
PAY_0,-0.030575,-0.271214,-0.039447,1.000000,0.672164,0.574245,0.538841,0.509426,0.474553,0.187068,0.189859,0.179785,0.179125,0.180635,0.176980,-0.079269,-0.070101,-0.070561,-0.064005,-0.058190,-0.058673
PAY_2,-0.011215,-0.296382,-0.050148,0.672164,1.000000,0.766552,0.662067,0.622780,0.575501,0.234887,0.235257,0.224146,0.222237,0.221348,0.219403,-0.080701,-0.058990,-0.055901,-0.046858,-0.037093,-0.036500
PAY_3,-0.018494,-0.286123,-0.053048,0.574245,0.766552,1.000000,0.777359,0.686775,0.632684,0.208473,0.237295,0.227494,0.227202,0.225145,0.222327,0.001295,-0.066793,-0.053311,-0.046067,-0.035863,-0.035861
PAY_4,-0.002735,-0.267460,-0.049722,0.538841,0.662067,0.777359,1.000000,0.819835,0.716449,0.202812,0.225816,0.244983,0.245917,0.242902,0.239154,-0.009362,-0.001944,-0.069235,-0.043461,-0.033590,-0.026565
PAY_5,-0.022199,-0.249411,-0.053826,0.509426,0.622780,0.686775,0.819835,1.000000,0.816900,0.206684,0.226913,0.243335,0.271915,0.269783,0.262509,-0.006089,-0.003191,0.009062,-0.058299,-0.033337,-0.023027
PAY_6,-0.020270,-0.235195,-0.048773,0.474553,0.575501,0.632684,0.716449,0.816900,1.000000,0.207373,0.226924,0.241181,0.266356,0.290894,0.285091,-0.001496,-0.005223,0.005834,0.019018,-0.046434,-0.025299
BILL_AMT1,0.019389,0.285430,0.056239,0.187068,0.234887,0.208473,0.202812,0.206684,0.207373,1.000000,0.951484,0.892279,0.860272,0.829779,0.802650,0.140277,0.099355,0.156887,0.158303,0.167026,0.179341


In [152]:
# Features
default_data = default.drop(columns = ['class'])
default_label = default['class']

default_non_cat_col = default_data.select_dtypes(exclude = 'category').columns
for c in default_non_cat_col:
    default_data[c] = pd.cut(default_data[c], bins=nbin, labels=range(nbin))
    
default_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
ID           30000 non-null category
LIMIT_BAL    30000 non-null category
SEX          30000 non-null category
EDUCATION    30000 non-null category
MARRIAGE     30000 non-null category
AGE          30000 non-null category
PAY_0        30000 non-null category
PAY_2        30000 non-null category
PAY_3        30000 non-null category
PAY_4        30000 non-null category
PAY_5        30000 non-null category
PAY_6        30000 non-null category
BILL_AMT1    30000 non-null category
BILL_AMT2    30000 non-null category
BILL_AMT3    30000 non-null category
BILL_AMT4    30000 non-null category
BILL_AMT5    30000 non-null category
BILL_AMT6    30000 non-null category
PAY_AMT1     30000 non-null category
PAY_AMT2     30000 non-null category
PAY_AMT3     30000 non-null category
PAY_AMT4     30000 non-null category
PAY_AMT5     30000 non-null category
PAY_AMT6     30000 non-null category
dty

In [153]:
default_data[default_data.isnull().any(axis=1)]

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6


In [154]:
default_data.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,0,0,2,2,1,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,0,0,2,2,2,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0
2,0,0,2,2,2,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
3,0,0,2,2,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,0,0,1,2,1,3,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0


In [155]:
default_label.head()

0    1
1    1
2    0
3    0
4    0
Name: class, dtype: category
Categories (2, int64): [0, 1]

In [156]:
default_label.cat.codes

0        1
1        1
2        0
3        0
4        0
        ..
29995    0
29996    0
29997    1
29998    1
29999    1
Length: 30000, dtype: int8

In [157]:
default_data.shape

(30000, 24)

In [158]:
with open('dataset/default-bin{}.pkl'.format(nbin), 'wb') as f:
    pickle.dump((default_data, pd.Series(default_label.cat.codes, name='class')), f)

# Poke

In [178]:
poke = pd.read_csv('dataset/poker-hand-testing.data', sep=",", engine='python', na_values=['?'])

In [179]:
poke.head()

,1,1.1,1.2,13,2,4,2.1,3,1.3,12,0
0,3,12,3,2,3,11,4,5,2,5,1
1,1,9,4,6,1,4,3,2,3,9,1
2,1,4,3,13,2,13,2,1,3,6,1
3,3,10,2,7,1,2,2,11,4,9,0
4,1,3,4,5,3,4,1,12,4,6,0


In [180]:
poke.columns = ['S1', 'C1', 'S2', 'C2', 'S3', 'C3', 'S4', 'C4', 'S5', 'C5'] + ['class']

In [181]:
poke['class']

0         1
1         1
2         1
3         0
4         0
         ..
999994    1
999995    1
999996    1
999997    1
999998    2
Name: class, Length: 999999, dtype: int64

In [182]:
poke['class'] = poke['class'].astype('category')

In [183]:
poke.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999999 entries, 0 to 999998
Data columns (total 11 columns):
S1       999999 non-null int64
C1       999999 non-null int64
S2       999999 non-null int64
C2       999999 non-null int64
S3       999999 non-null int64
C3       999999 non-null int64
S4       999999 non-null int64
C4       999999 non-null int64
S5       999999 non-null int64
C5       999999 non-null int64
class    999999 non-null category
dtypes: category(1), int64(10)
memory usage: 77.2 MB


In [184]:
poke_obj_col = poke.columns[:-1].tolist()
for c in poke_obj_col:
    poke[c] = poke[c].astype('category')

In [185]:
poke_obj_col = poke.select_dtypes('category').columns
print([len(poke[c].cat.categories) for c in poke_obj_col])

[4, 13, 4, 13, 4, 13, 4, 13, 4, 13, 10]


In [87]:
# poke = poke.drop(columns = ['diag_1','diag_2','diag_3','medical_specialty'])

In [186]:
poke.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999999 entries, 0 to 999998
Data columns (total 11 columns):
S1       999999 non-null category
C1       999999 non-null category
S2       999999 non-null category
C2       999999 non-null category
S3       999999 non-null category
C3       999999 non-null category
S4       999999 non-null category
C4       999999 non-null category
S5       999999 non-null category
C5       999999 non-null category
class    999999 non-null category
dtypes: category(11)
memory usage: 10.5 MB


In [187]:
poke.describe()

,S1,C1,S2,C2,S3,C3,S4,C4,S5,C5,class
count,999999,999999,999999,999999,999999,999999,999999,999999,999999,999999,999999
unique,4,13,4,13,4,13,4,13,4,13,10
top,3,6,1,11,3,7,3,7,1,3,0
freq,250900,77282,250361,77373,250748,77517,250600,77581,250754,77566,501208


In [188]:
poke.select_dtypes('category').describe()

,S1,C1,S2,C2,S3,C3,S4,C4,S5,C5,class
count,999999,999999,999999,999999,999999,999999,999999,999999,999999,999999,999999
unique,4,13,4,13,4,13,4,13,4,13,10
top,3,6,1,11,3,7,3,7,1,3,0
freq,250900,77282,250361,77373,250748,77517,250600,77581,250754,77566,501208


In [189]:
display(poke.corr())

""


In [190]:
# Features
poke_data = poke.drop(columns = ['class'])
poke_label = poke['class']

poke_non_cat_col = poke_data.select_dtypes(exclude = 'category').columns
for c in poke_non_cat_col:
    poke_data[c] = pd.cut(poke_data[c], bins=nbin, labels=range(nbin))
    
poke_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999999 entries, 0 to 999998
Data columns (total 10 columns):
S1    999999 non-null category
C1    999999 non-null category
S2    999999 non-null category
C2    999999 non-null category
S3    999999 non-null category
C3    999999 non-null category
S4    999999 non-null category
C4    999999 non-null category
S5    999999 non-null category
C5    999999 non-null category
dtypes: category(10)
memory usage: 9.5 MB


In [191]:
poke_data[poke_data.isnull().any(axis=1)]

,S1,C1,S2,C2,S3,C3,S4,C4,S5,C5


In [192]:
poke_data.head()

,S1,C1,S2,C2,S3,C3,S4,C4,S5,C5
0,3,12,3,2,3,11,4,5,2,5
1,1,9,4,6,1,4,3,2,3,9
2,1,4,3,13,2,13,2,1,3,6
3,3,10,2,7,1,2,2,11,4,9
4,1,3,4,5,3,4,1,12,4,6


In [193]:
poke_label.head()

0    1
1    1
2    1
3    0
4    0
Name: class, dtype: category
Categories (10, int64): [0, 1, 2, 3, ..., 6, 7, 8, 9]

In [194]:
poke_label.cat.codes

0         1
1         1
2         1
3         0
4         0
         ..
999994    1
999995    1
999996    1
999997    1
999998    2
Length: 999999, dtype: int8

In [195]:
poke_data.shape

(999999, 10)

In [196]:
with open('dataset/poke-bin{}.pkl'.format(nbin), 'wb') as f:
    pickle.dump((poke_data, pd.Series(poke_label.cat.codes, name='class')), f)

# Dota2

In [1]:
dota = pd.read_csv('dataset/dota2Train.csv', sep=",", engine='python', na_values=['?'])

In [2]:
dota.head()

,-1,223,2,2.1,0,0.1,0.2,0.3,0.4,0.5,...,0.93,0.94,0.95,0.96,0.97,0.98,0.99,0.100,0.101,0.102
0,1,152,2,2,0,0,0,1,0,-1,...,0,0,0,0,0,0,0,0,0,0
1,1,131,2,2,0,0,0,1,0,-1,...,0,0,0,0,0,0,0,0,0,0
2,1,154,2,2,0,0,0,0,0,0,...,-1,0,0,0,0,0,0,0,0,0
3,-1,171,2,3,0,0,0,0,0,-1,...,0,0,0,0,0,0,0,0,0,0
4,1,122,2,3,0,1,0,0,0,0,...,1,0,0,0,0,0,0,0,0,-1


In [3]:
dota.columns = ['class', 'clusterID', 'mode', 'type'] + ['hero{}'.format(i) for i in range(113)]

In [4]:
dota['class']

0        1
1        1
2        1
3       -1
4        1
        ..
92644   -1
92645    1
92646    1
92647   -1
92648   -1
Name: class, Length: 92649, dtype: int64

In [5]:
dota['class'] = dota['class'].astype('category')

In [6]:
dota.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92649 entries, 0 to 92648
Columns: 117 entries, class to hero112
dtypes: category(1), int64(116)
memory usage: 82.1 MB


In [7]:
# dota_obj_col = dota.columns[:-1].tolist()
dota_obj_col = ['mode', 'type'] + ['hero{}'.format(i) for i in range(113)]
for c in dota_obj_col:
    dota[c] = dota[c].astype('category')

In [8]:
dota_obj_col = dota.select_dtypes('category').columns
print([len(dota[c].cat.categories) for c in dota_obj_col])

[2, 9, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3]


In [87]:
# dota = dota.drop(columns = ['diag_1','diag_2','diag_3','medical_specialty'])

In [9]:
dota.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92649 entries, 0 to 92648
Columns: 117 entries, class to hero112
dtypes: category(116), int64(1)
memory usage: 11.0 MB


In [10]:
dota.describe()

,clusterID
count,92649.000000
mean,175.863636
std,35.658070
min,111.000000
25%,152.000000
50%,156.000000
75%,223.000000
max,261.000000


In [11]:
dota.select_dtypes('category').describe()

,class,mode,type,hero0,hero1,hero2,hero3,hero4,hero5,hero6,...,hero103,hero104,hero105,hero106,hero107,hero108,hero109,hero110,hero111,hero112
count,92649,92649,92649,92649,92649,92649,92649,92649,92649,92649,...,92649,92649,92649,92649,92649,92649,92649,92649,92649,92649
unique,2,9,3,3,3,3,3,3,3,3,...,3,3,3,3,1,3,3,3,3,3
top,1,2,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
freq,48782,71896,56987,77676,72385,90125,80947,82599,70949,81373,...,66128,88713,85177,90396,92649,88136,88787,90012,89309,90858


In [12]:
display(dota.corr())

,clusterID
clusterID,1.0


In [17]:
# Features
dota_data = dota.drop(columns = ['class'])
dota_label = dota['class']

dota_non_cat_col = dota_data.select_dtypes(exclude = 'category').columns
for c in dota_non_cat_col:
    dota_data[c] = pd.cut(dota_data[c], bins=nbin, labels=range(nbin))
    
dota_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92649 entries, 0 to 92648
Columns: 116 entries, clusterID to hero112
dtypes: category(116)
memory usage: 10.3 MB


In [18]:
dota_data[dota_data.isnull().any(axis=1)]

,clusterID,mode,type,hero0,hero1,hero2,hero3,hero4,hero5,hero6,hero7,hero8,hero9,hero10,hero11,hero12,hero13,hero14,hero15,hero16,hero17,hero18,hero19,hero20,hero21,hero22,hero23,hero24,hero25,hero26,hero27,hero28,hero29,hero30,hero31,hero32,hero33,hero34,hero35,hero36,hero37,hero38,hero39,hero40,hero41,hero42,hero43,hero44,hero45,hero46,hero47,hero48,hero49,hero50,hero51,hero52,hero53,hero54,hero55,hero56,hero57,hero58,hero59,hero60,hero61,hero62,hero63,hero64,hero65,hero66,hero67,hero68,hero69,hero70,hero71,hero72,hero73,hero74,hero75,hero76,hero77,hero78,hero79,hero80,hero81,hero82,hero83,hero84,hero85,hero86,hero87,hero88,hero89,hero90,hero91,hero92,hero93,hero94,hero95,hero96,hero97,hero98,hero99,hero100,hero101,hero102,hero103,hero104,hero105,hero106,hero107,hero108,hero109,hero110,hero111,hero112


In [19]:
dota_data.head()

,clusterID,mode,type,hero0,hero1,hero2,hero3,hero4,hero5,hero6,hero7,hero8,hero9,hero10,hero11,hero12,hero13,hero14,hero15,hero16,hero17,hero18,hero19,hero20,hero21,hero22,hero23,hero24,hero25,hero26,hero27,hero28,hero29,hero30,hero31,hero32,hero33,hero34,hero35,hero36,hero37,hero38,hero39,hero40,hero41,hero42,hero43,hero44,hero45,hero46,hero47,hero48,hero49,hero50,hero51,hero52,hero53,hero54,hero55,hero56,hero57,hero58,hero59,hero60,hero61,hero62,hero63,hero64,hero65,hero66,hero67,hero68,hero69,hero70,hero71,hero72,hero73,hero74,hero75,hero76,hero77,hero78,hero79,hero80,hero81,hero82,hero83,hero84,hero85,hero86,hero87,hero88,hero89,hero90,hero91,hero92,hero93,hero94,hero95,hero96,hero97,hero98,hero99,hero100,hero101,hero102,hero103,hero104,hero105,hero106,hero107,hero108,hero109,hero110,hero111,hero112
0,1,2,2,0,0,0,1,0,-1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,-1,0,0,0,0,1,1,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,2,2,0,0,0,1,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,2,2,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,-1,0,0,0,0,-1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0
3,1,2,3,0,0,0,0,0,-1,0,0,-1,0,-1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,2,3,0,1,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,-1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,-1


In [20]:
dota_label.head()

0    1
1    1
2    1
3   -1
4    1
Name: class, dtype: category
Categories (2, int64): [-1, 1]

In [21]:
dota_label.cat.codes

0        1
1        1
2        1
3        0
4        1
        ..
92644    0
92645    1
92646    1
92647    0
92648    0
Length: 92649, dtype: int8

In [22]:
dota_data.shape

(92649, 116)

In [23]:
with open('dataset/dota-bin{}.pkl'.format(nbin), 'wb') as f:
    pickle.dump((dota_data, pd.Series(dota_label.cat.codes, name='class')), f)

# Firewall

In [24]:
wall = pd.read_csv('dataset/firewall.csv', sep=",", engine='python', na_values=['?'])

In [25]:
wall.head()

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Action,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received
0,57222,53,54587,53,allow,177,94,83,2,30,1,1
1,56258,3389,56258,3389,allow,4768,1600,3168,19,17,10,9
2,6881,50321,43265,50321,allow,238,118,120,2,1199,1,1
3,50553,3389,50553,3389,allow,3327,1438,1889,15,17,8,7
4,50002,443,45848,443,allow,25358,6778,18580,31,16,13,18


In [27]:
wall.columns = ['Source Port', 'Destination Port', 'NAT Source Port',
       'NAT Destination Port', 'class', 'Bytes', 'Bytes Sent',
       'Bytes Received', 'Packets', 'Elapsed Time (sec)', 'pkts_sent',
       'pkts_received']

In [28]:
wall['class']

0        allow
1        allow
2        allow
3        allow
4        allow
         ...  
65527    allow
65528    allow
65529     drop
65530     drop
65531     drop
Name: class, Length: 65532, dtype: object

In [29]:
wall['class'] = wall['class'].astype('category')

In [ ]:
wall['class'].value_counts()

In [30]:
wall.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65532 entries, 0 to 65531
Data columns (total 12 columns):
Source Port             65532 non-null int64
Destination Port        65532 non-null int64
NAT Source Port         65532 non-null int64
NAT Destination Port    65532 non-null int64
class                   65532 non-null category
Bytes                   65532 non-null int64
Bytes Sent              65532 non-null int64
Bytes Received          65532 non-null int64
Packets                 65532 non-null int64
Elapsed Time (sec)      65532 non-null int64
pkts_sent               65532 non-null int64
pkts_received           65532 non-null int64
dtypes: category(1), int64(11)
memory usage: 5.6 MB


In [7]:
# wall_obj_col = wall.columns[:-1].tolist()
# wall_obj_col = ['mode', 'type'] + ['hero{}'.format(i) for i in range(113)]
# for c in wall_obj_col:
#     wall[c] = wall[c].astype('category')

In [31]:
wall_obj_col = wall.select_dtypes('category').columns
print([len(wall[c].cat.categories) for c in wall_obj_col])

[4]


In [87]:
# wall = wall.drop(columns = ['diag_1','diag_2','diag_3','medical_specialty'])

In [32]:
wall.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65532 entries, 0 to 65531
Data columns (total 12 columns):
Source Port             65532 non-null int64
Destination Port        65532 non-null int64
NAT Source Port         65532 non-null int64
NAT Destination Port    65532 non-null int64
class                   65532 non-null category
Bytes                   65532 non-null int64
Bytes Sent              65532 non-null int64
Bytes Received          65532 non-null int64
Packets                 65532 non-null int64
Elapsed Time (sec)      65532 non-null int64
pkts_sent               65532 non-null int64
pkts_received           65532 non-null int64
dtypes: category(1), int64(11)
memory usage: 5.6 MB


In [33]:
wall.describe()

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received
count,65532.000000,65532.000000,65532.000000,65532.000000,6.553200e+04,6.553200e+04,6.553200e+04,6.553200e+04,65532.000000,65532.000000,65532.000000
mean,49391.969343,10577.385812,19282.972761,2671.049930,9.712395e+04,2.238580e+04,7.473815e+04,1.028660e+02,65.833577,41.399530,61.466505
std,15255.712537,18466.027039,21970.689669,9739.162278,5.618439e+06,3.828139e+06,2.463208e+06,5.133002e+03,302.461762,3218.871288,2223.332271
min,0.000000,0.000000,0.000000,0.000000,6.000000e+01,6.000000e+01,0.000000e+00,1.000000e+00,0.000000,1.000000,0.000000
25%,49183.000000,80.000000,0.000000,0.000000,6.600000e+01,6.600000e+01,0.000000e+00,1.000000e+00,0.000000,1.000000,0.000000
50%,53776.500000,445.000000,8820.500000,53.000000,1.680000e+02,9.000000e+01,7.900000e+01,2.000000e+00,15.000000,1.000000,1.000000
75%,58638.000000,15000.000000,38366.250000,443.000000,7.522500e+02,2.100000e+02,4.490000e+02,6.000000e+00,30.000000,3.000000,2.000000
max,65534.000000,65535.000000,65535.000000,65535.000000,1.269359e+09,9.484772e+08,3.208818e+08,1.036116e+06,10824.000000,747520.000000,327208.000000


In [34]:
wall.select_dtypes('category').describe()

,class
count,65532
unique,4
top,allow
freq,37640


In [35]:
display(wall.corr())

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received
Source Port,1.000000,-0.332246,0.145391,-0.024843,0.000221,-0.000931,0.001950,-0.001742,-0.046515,-0.001422,-0.001962
Destination Port,-0.332246,1.000000,-0.281676,0.410042,-0.005297,0.001675,-0.014684,-0.006063,0.023537,-0.002134,-0.010909
NAT Source Port,0.145391,-0.281676,1.000000,0.178435,0.010659,0.002242,0.020827,0.012633,0.141485,0.007180,0.018772
NAT Destination Port,-0.024843,0.410042,0.178435,1.000000,0.003975,0.007904,-0.003216,0.004605,0.219776,0.006136,0.001747
Bytes,0.000221,-0.005297,0.010659,0.003975,1.000000,0.933462,0.830225,0.974379,0.148834,0.966548,0.850209
Bytes Sent,-0.000931,0.001675,0.002242,0.007904,0.933462,1.000000,0.575047,0.887596,0.126039,0.973976,0.639098
Bytes Received,0.001950,-0.014684,0.020827,-0.003216,0.830225,0.575047,1.000000,0.843067,0.143601,0.690959,0.946039
Packets,-0.001742,-0.006063,0.012633,0.004605,0.974379,0.887596,0.843067,1.000000,0.147074,0.961286,0.916978
Elapsed Time (sec),-0.046515,0.023537,0.141485,0.219776,0.148834,0.126039,0.143601,0.147074,1.000000,0.135101,0.143954
pkts_sent,-0.001422,-0.002134,0.007180,0.006136,0.966548,0.973976,0.690959,0.961286,0.135101,1.000000,0.771550


In [36]:
# Features
wall_data = wall.drop(columns = ['class'])
wall_label = wall['class']

wall_non_cat_col = wall_data.select_dtypes(exclude = 'category').columns
for c in wall_non_cat_col:
    wall_data[c] = pd.cut(wall_data[c], bins=nbin, labels=range(nbin))
    
wall_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65532 entries, 0 to 65531
Data columns (total 11 columns):
Source Port             65532 non-null category
Destination Port        65532 non-null category
NAT Source Port         65532 non-null category
NAT Destination Port    65532 non-null category
Bytes                   65532 non-null category
Bytes Sent              65532 non-null category
Bytes Received          65532 non-null category
Packets                 65532 non-null category
Elapsed Time (sec)      65532 non-null category
pkts_sent               65532 non-null category
pkts_received           65532 non-null category
dtypes: category(11)
memory usage: 705.5 KB


In [37]:
wall_data[wall_data.isnull().any(axis=1)]

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received


In [38]:
wall_data.head()

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received
0,4,0,4,0,0,0,0,0,0,0,0
1,4,0,4,0,0,0,0,0,0,0,0
2,0,3,3,3,0,0,0,0,0,0,0
3,3,0,3,0,0,0,0,0,0,0,0
4,3,0,3,0,0,0,0,0,0,0,0


In [39]:
wall_label.head()

0    allow
1    allow
2    allow
3    allow
4    allow
Name: class, dtype: category
Categories (4, object): [allow, deny, drop, reset-both]

In [40]:
wall_label.cat.codes

0        0
1        0
2        0
3        0
4        0
        ..
65527    0
65528    0
65529    2
65530    2
65531    2
Length: 65532, dtype: int8

In [41]:
wall_data.shape

(65532, 11)

In [42]:
with open('dataset/firewall-bin{}.pkl'.format(nbin), 'wb') as f:
    pickle.dump((wall_data, pd.Series(wall_label.cat.codes, name='class')), f)

# Obesity

In [44]:
obesity = pd.read_csv('dataset/ObesityDataSet_raw_and_data_sinthetic.csv', sep=",", engine='python', na_values=['?'])

In [45]:
obesity.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [50]:
obesity.columns = obesity.columns[:-1].to_list() + ['class']

In [51]:
obesity['class']

0             Normal_Weight
1             Normal_Weight
2             Normal_Weight
3        Overweight_Level_I
4       Overweight_Level_II
               ...         
2106       Obesity_Type_III
2107       Obesity_Type_III
2108       Obesity_Type_III
2109       Obesity_Type_III
2110       Obesity_Type_III
Name: class, Length: 2111, dtype: object

In [52]:
obesity['class'] = obesity['class'].astype('category')

In [53]:
obesity['class'].value_counts()

Obesity_Type_I         351
Obesity_Type_III       324
Obesity_Type_II        297
Overweight_Level_II    290
Overweight_Level_I     290
Normal_Weight          287
Insufficient_Weight    272
Name: class, dtype: int64

In [54]:
obesity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2111 entries, 0 to 2110
Data columns (total 17 columns):
Gender                            2111 non-null object
Age                               2111 non-null float64
Height                            2111 non-null float64
Weight                            2111 non-null float64
family_history_with_overweight    2111 non-null object
FAVC                              2111 non-null object
FCVC                              2111 non-null float64
NCP                               2111 non-null float64
CAEC                              2111 non-null object
SMOKE                             2111 non-null object
CH2O                              2111 non-null float64
SCC                               2111 non-null object
FAF                               2111 non-null float64
TUE                               2111 non-null float64
CALC                              2111 non-null object
MTRANS                            2111 non-null object
class

In [55]:
obesity_obj_col = obesity.select_dtypes('object')
for c in obesity_obj_col:
    obesity[c] = obesity[c].astype('category')

In [56]:
obesity_obj_col = obesity.select_dtypes('category').columns
print([len(obesity[c].cat.categories) for c in obesity_obj_col])

[2, 2, 2, 4, 2, 2, 4, 5, 7]


In [87]:
# obesity = obesity.drop(columns = ['diag_1','diag_2','diag_3','medical_specialty'])

In [57]:
obesity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2111 entries, 0 to 2110
Data columns (total 17 columns):
Gender                            2111 non-null category
Age                               2111 non-null float64
Height                            2111 non-null float64
Weight                            2111 non-null float64
family_history_with_overweight    2111 non-null category
FAVC                              2111 non-null category
FCVC                              2111 non-null float64
NCP                               2111 non-null float64
CAEC                              2111 non-null category
SMOKE                             2111 non-null category
CH2O                              2111 non-null float64
SCC                               2111 non-null category
FAF                               2111 non-null float64
TUE                               2111 non-null float64
CALC                              2111 non-null category
MTRANS                            2111 non-nul

In [58]:
obesity.describe()

,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE
count,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000,2111.000000
mean,24.312600,1.701677,86.586058,2.419043,2.685628,2.008011,1.010298,0.657866
std,6.345968,0.093305,26.191172,0.533927,0.778039,0.612953,0.850592,0.608927
min,14.000000,1.450000,39.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,19.947192,1.630000,65.473343,2.000000,2.658738,1.584812,0.124505,0.000000
50%,22.777890,1.700499,83.000000,2.385502,3.000000,2.000000,1.000000,0.625350
75%,26.000000,1.768464,107.430682,3.000000,3.000000,2.477420,1.666678,1.000000
max,61.000000,1.980000,173.000000,3.000000,4.000000,3.000000,3.000000,2.000000


In [59]:
obesity.select_dtypes('category').describe()

,Gender,family_history_with_overweight,FAVC,CAEC,SMOKE,SCC,CALC,MTRANS,class
count,2111,2111,2111,2111,2111,2111,2111,2111,2111
unique,2,2,2,4,2,2,4,5,7
top,Male,yes,yes,Sometimes,no,no,Sometimes,Public_Transportation,Obesity_Type_I
freq,1068,1726,1866,1765,2067,2015,1401,1580,351


In [60]:
display(obesity.corr())

,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE
Age,1.000000,-0.025958,0.202560,0.016291,-0.043944,-0.045304,-0.144938,-0.296931
Height,-0.025958,1.000000,0.463136,-0.038121,0.243672,0.213376,0.294709,0.051912
Weight,0.202560,0.463136,1.000000,0.216125,0.107469,0.200575,-0.051436,-0.071561
FCVC,0.016291,-0.038121,0.216125,1.000000,0.042216,0.068461,0.019939,-0.101135
NCP,-0.043944,0.243672,0.107469,0.042216,1.000000,0.057088,0.129504,0.036326
CH2O,-0.045304,0.213376,0.200575,0.068461,0.057088,1.000000,0.167236,0.011965
FAF,-0.144938,0.294709,-0.051436,0.019939,0.129504,0.167236,1.000000,0.058562
TUE,-0.296931,0.051912,-0.071561,-0.101135,0.036326,0.011965,0.058562,1.000000


In [61]:
# Features
obesity_data = obesity.drop(columns = ['class'])
obesity_label = obesity['class']

obesity_non_cat_col = obesity_data.select_dtypes(exclude = 'category').columns
for c in obesity_non_cat_col:
    obesity_data[c] = pd.cut(obesity_data[c], bins=nbin, labels=range(nbin))
    
obesity_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2111 entries, 0 to 2110
Data columns (total 16 columns):
Gender                            2111 non-null category
Age                               2111 non-null category
Height                            2111 non-null category
Weight                            2111 non-null category
family_history_with_overweight    2111 non-null category
FAVC                              2111 non-null category
FCVC                              2111 non-null category
NCP                               2111 non-null category
CAEC                              2111 non-null category
SMOKE                             2111 non-null category
CH2O                              2111 non-null category
SCC                               2111 non-null category
FAF                               2111 non-null category
TUE                               2111 non-null category
CALC                              2111 non-null category
MTRANS                            2111

In [62]:
obesity_data[obesity_data.isnull().any(axis=1)]

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS


In [63]:
obesity_data.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS
0,Female,0,1,0,yes,no,2,3,Sometimes,no,2,no,0,2,no,Public_Transportation
1,Female,0,0,0,yes,no,4,3,Sometimes,yes,4,yes,4,0,Sometimes,Public_Transportation
2,Male,0,3,1,yes,no,2,3,Sometimes,no,2,no,3,2,Frequently,Public_Transportation
3,Male,1,3,1,no,no,4,3,Sometimes,no,2,no,3,0,Frequently,Walking
4,Male,0,3,1,no,no,2,0,Sometimes,no,2,no,0,0,Sometimes,Public_Transportation


In [64]:
obesity_label.head()

0          Normal_Weight
1          Normal_Weight
2          Normal_Weight
3     Overweight_Level_I
4    Overweight_Level_II
Name: class, dtype: category
Categories (7, object): [Insufficient_Weight, Normal_Weight, Obesity_Type_I, Obesity_Type_II, Obesity_Type_III, Overweight_Level_I, Overweight_Level_II]

In [65]:
obesity_label.cat.codes

0       1
1       1
2       1
3       5
4       6
       ..
2106    4
2107    4
2108    4
2109    4
2110    4
Length: 2111, dtype: int8

In [41]:
obesity_data.shape

(65532, 11)

In [66]:
with open('dataset/obesity-bin{}.pkl'.format(nbin), 'wb') as f:
    pickle.dump((obesity_data, pd.Series(obesity_label.cat.codes, name='class')), f)

# FBNode

In [141]:
fb = pd.read_csv('dataset/facebook_large/musae_facebook_target.csv', sep=",", engine='python', na_values=['?'])

In [142]:
fb.head()

,id,facebook_id,page_name,page_type
0,0,145647315578475,The Voice of China 中国好声音,tvshow
1,1,191483281412,U.S. Consulate General Mumbai,government
2,2,144761358898518,ESET,company
3,3,568700043198473,Consulate General of Switzerland in Montreal,government
4,4,1408935539376139,Mark Bailey MP - Labor for Miller,politician


In [143]:
fb = fb.drop(columns = ['facebook_id', 'page_name'])

In [144]:
import json
with open('dataset/facebook_large/musae_facebook_features.json') as json_file:
    feat = json.load(json_file)

In [145]:
nfeat = 4714
arrs = []
for u in range(len(feat)):
    arr = np.zeros(nfeat, dtype=np.bool)
    for i in feat[str(u)]:
        arr[i] = 1
    arr = pd.Series(arr, dtype="category")
    arrs.append(arr)

In [146]:
dff = pd.DataFrame(arrs, columns=[u for u in range(nfeat)], dtype="category")

In [147]:
dff.shape

(22470, 4714)

In [148]:
fb = pd.concat([fb,dff], axis=1)
fb.shape

(22470, 4716)

In [149]:
fb = fb.drop(columns = ['id'])

In [150]:
fb.columns = ['class'] + fb.columns[1:].to_list()

In [151]:
fb['class']

0            tvshow
1        government
2           company
3        government
4        politician
            ...    
22465    politician
22466       company
22467    government
22468       company
22469        tvshow
Name: class, Length: 22470, dtype: object

In [152]:
fb['class'] = fb['class'].astype('category')

In [153]:
fb['class'].value_counts()

government    6880
company       6495
politician    5768
tvshow        3327
Name: class, dtype: int64

In [154]:
fb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22470 entries, 0 to 22469
Columns: 4715 entries, class to 4713
dtypes: category(4715)
memory usage: 101.5 MB


In [155]:
fb.dtypes

class    category
0        category
1        category
2        category
3        category
           ...   
4709     category
4710     category
4711     category
4712     category
4713     category
Length: 4715, dtype: object

In [156]:
# fb_obj_col = fb.select_dtypes('object')
# for c in fb_obj_col:
#     fb[c] = fb[c].astype('category')

In [157]:
fb_obj_col = fb.select_dtypes('category').columns
print([len(fb[c].cat.categories) for c in fb_obj_col])

[4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 

In [87]:
# fb = fb.drop(columns = ['diag_1','diag_2','diag_3','medical_specialty'])

In [158]:
fb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22470 entries, 0 to 22469
Columns: 4715 entries, class to 4713
dtypes: category(4715)
memory usage: 101.5 MB


In [159]:
fb.describe()

class      0      1      2      3      4      5      6      7  \
count        22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique           4      2      2      2      2      2      2      2      2   
top     government  False  False  False  False  False  False  False  False   
freq          6880  22462  22462  22444  22465  22468  22458  22465  22468   

            8      9     10     11     12     13     14     15     16     17  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22440  22460  22429  22465  22462  22430  22459  22468  22352  22465   

           18     19     20     21     22     23     24     25     26     27  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22463  22468  22468  22050  22465  22460  22440  22468  22452  22465   

           28     29     30     31     32     33     34     35     36     37  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22466  22447  22466  22468  22463  22454  22468  22379  22459  22249   

           38     39     40     41     42     43     44     45     46     47  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22467  22463  22378  22447  22462  22460  22461  22447  22466  22467   

           48     49     50     51     52     53     54     55     56     57  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22416  22342  22469  22456  22468  22467  22460  22469  22458  22443   

           58     59     60     61     62     63     64     65     66     67  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22454  22468  22467  22457  22464  22447  22468  22469  22284  22457   

           68     69     70     71     72     73     74     75     76     77  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22448  22407  22431  22450  22465  22466  22286  22466  22465  22453   

           78     79     80     81     82     83     84     85     86     87  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22405  22468  22068  22459  22465  22374  22468  22453  22464  22468   

           88     89     90     91     92     93     94     95     96     97  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22464  22463  22468  22463  22429  22467  22467  22465  22465  22412   

           98

In [160]:
fb.select_dtypes('category').describe()

class      0      1      2      3      4      5      6      7  \
count        22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique           4      2      2      2      2      2      2      2      2   
top     government  False  False  False  False  False  False  False  False   
freq          6880  22462  22462  22444  22465  22468  22458  22465  22468   

            8      9     10     11     12     13     14     15     16     17  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22440  22460  22429  22465  22462  22430  22459  22468  22352  22465   

           18     19     20     21     22     23     24     25     26     27  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22463  22468  22468  22050  22465  22460  22440  22468  22452  22465   

           28     29     30     31     32     33     34     35     36     37  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22466  22447  22466  22468  22463  22454  22468  22379  22459  22249   

           38     39     40     41     42     43     44     45     46     47  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22467  22463  22378  22447  22462  22460  22461  22447  22466  22467   

           48     49     50     51     52     53     54     55     56     57  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22416  22342  22469  22456  22468  22467  22460  22469  22458  22443   

           58     59     60     61     62     63     64     65     66     67  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22454  22468  22467  22457  22464  22447  22468  22469  22284  22457   

           68     69     70     71     72     73     74     75     76     77  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22448  22407  22431  22450  22465  22466  22286  22466  22465  22453   

           78     79     80     81     82     83     84     85     86     87  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22405  22468  22068  22459  22465  22374  22468  22453  22464  22468   

           88     89     90     91     92     93     94     95     96     97  \
count   22470  22470  22470  22470  22470  22470  22470  22470  22470  22470   
unique      2      2      2      2      2      2      2      2      2      2   
top     False  False  False  False  False  False  False  False  False  False   
freq    22464  22463  22468  22463  22429  22467  22467  22465  22465  22412   

           98

In [161]:
display(fb.corr())

""


In [171]:
fb = fb.sample(frac=0.1, axis=0)

In [172]:
# Features
fb_data = fb.drop(columns = ['class'])
fb_label = fb['class']

fb_non_cat_col = fb_data.select_dtypes(exclude = 'category').columns
for c in fb_non_cat_col:
    fb_data[c] = pd.cut(fb_data[c], bins=nbin, labels=range(nbin))
    
fb_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2247 entries, 11264 to 11788
Columns: 4714 entries, 0 to 4713
dtypes: category(4714)
memory usage: 10.6 MB


In [173]:
fb_data[fb_data.isnull().any(axis=1)]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,622,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,696,697,698,699,700,701,702,703,704,705,706,707,708,709,710,711,712,713,714,715,716,717,718,719,720,721,722,723,724,725,726,727,728,729,730,731,732,733,734,735,736,737,738,739,740,741,742,743,744,745,746,747,748,749,750,751,752,753,754,755,756,757,758,759,760,761,762,763,764,765,766,767,768,769,770,771,772,773,774,775,776,777,778,779,780,781,782,783,784,785,786,787,788,789,790,791,792,793,794,795,796,797,798,799,800,801,802,803,804,805,806,807,808,809,810,811,812,813,814,815,816,817,818,819,820,821,822,823,824,825,826,827,828,829,830,831,832,833,834,835,836,837,838,839,840,841,842,843,844,845,846,847,848,849,850,851,852,853,854,855,856,857,858,859,860,861,862,863,864,865,866,867,868,869,870,871,872,873,874,875,876,877,878,879,880,881,882,883,884,885,886,887,888,889,890,891,892,893,894,895,896,897,898,899,900,901,902,903,904,905,906,907,908,909,910,911,912,913,914,915,916,917,918,919,920,921,922,923,924,925,926,927,928,929,930,931,932,933,934,935,936,937,938,939,940,941,942,943,944,945,946,947,948,949,950,951,952,953,954,955,956,957,958,959,960,961,962,963,964,965,966,967,968,969,970,971,972,973,974,975,976,977,978,979,980,981,982,983,984,985,986,987,988,989,990,991,992,993,994,995,996,997,998,999,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021

In [174]:
fb_data.head()

0      1      2      3      4      5      6      7      8      9     \
11264  False  False  False  False  False  False  False  False  False  False   
18479  False  False  False  False  False  False  False  False  False  False   
15354  False  False  False  False  False  False  False  False  False  False   
16424  False  False  False  False  False  False  False  False  False  False   
11084  False  False  False  False  False  False  False  False  False  False   

        10     11     12     13     14     15     16     17     18     19    \
11264  False  False  False  False  False  False  False  False  False  False   
18479  False  False  False  False  False  False  False  False  False  False   
15354  False  False  False  False  False  False  False  False  False  False   
16424  False  False  False  False  False  False  False  False  False  False   
11084  False  False  False  False  False  False  False  False  False  False   

        20     21     22     23     24     25     26     27     28     29    \
11264  False  False  False  False  False  False  False  False  False  False   
18479  False  False  False  False  False  False  False  False  False  False   
15354  False  False  False  False  False  False  False  False  False  False   
16424  False  False  False  False  False  False  False  False  False  False   
11084  False  False  False  False  False  False  False  False  False  False   

        30     31     32     33     34     35     36     37     38     39    \
11264  False  False  False  False  False  False  False  False  False  False   
18479  False  False  False  False  False  False  False  False  False  False   
15354  False  False  False  False  False  False  False  False  False  False   
16424  False  False  False  False  False  False  False  False  False  False   
11084  False  False  False  False  False  False  False  False  False  False   

        40     41     42     43     44     45     46     47     48     49    \
11264  False  False  False  False  False  False  False  False  False  False   
18479  False  False  False  False  False  False  False  False  False  False   
15354  False  False  False  False  False  False  False  False  False  False   
16424  False  False  False  False  False  False  False  False  False  False   
11084  False  False  False  False  False  False  False  False  False  False   

        50     51     52     53     54     55     56     57     58     59    \
11264  False  False  False  False  False  False  False  False  False  False   
18479  False  False  False  False  False  False  False  False  False  False   
15354  False  False  False  False  False  False  False  False  False  False   
16424  False  False  False  False  False  False  False  False  False  False   
11084  False  False  False  False  False  False  False  False  False  False   

        60     61     62     63     64     65     66     67     68     69    \
11264  False  False  False  False  False  False  False  False  False  False   
18479  False  False  False  False  False  False  False  False  False  False   
15354  False  False  False  False  False  False  False  False  False  False   
16424  False  False  False  False  False  False  False  False  False  False   
11084  False  False  False  False  False  False  False  False  False  False   

        70     71     72     73     74     75     76     77     78     79    \
11264  False  False  False  False  False  False  False  False  False  False   
18479  False  False  False  False  False  False  False  False  False  False   
15354  False  False  False  False  False  False  False  False  False  False   
16424  False  False  False  False  False  False  False  False  False  False   
11084  False  False  False  False  False  False  False  False  False  False   

        80     81     82     83     84     85     86     87     88     89    \
11264  False  False  False  False  False  False  False  False  False  False   
18479  False  False  False  False  False  False  F

In [175]:
fb_label.head()

11264        tvshow
18479        tvshow
15354       company
16424    politician
11084    government
Name: class, dtype: category
Categories (4, object): [company, government, politician, tvshow]

In [176]:
fb_label.cat.codes

11264    3
18479    3
15354    0
16424    2
11084    1
        ..
13979    0
10702    1
3189     2
8376     1
11788    0
Length: 2247, dtype: int8

In [177]:
fb_data.shape

(2247, 4714)

In [178]:
with open('dataset/fb-bin{}.pkl'.format(nbin), 'wb') as f:
    pickle.dump((fb_data, pd.Series(fb_label.cat.codes, name='class')), f)

# STOP